# Recodificación: Encuesta 2026.sav

Este notebook carga el archivo SPSS, muestra las etiquetas de variables y valores, y aplica recodificaciones estándar para facilitar el análisis.

**Estructura del archivo:**
- 5.000 casos
- 587 variables
- Factores de expansión: `FE_HOGAR` y `FE_PERSONAS`
- Ponderadores: `POND_HOGAR` y `POND_PERSONAS`

## 1. Carga del archivo

In [ ]:
import pyreadstat
import pandas as pd
import numpy as np

# Carga el archivo .sav y su metadata
df, meta = pyreadstat.read_sav("2026.sav")

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

## 2. Diccionario de variables

Genera un DataFrame con el nombre de cada variable y su etiqueta (equivalente al 'label' en SPSS).

In [ ]:
# Crea un diccionario de variables con nombre y etiqueta
diccionario = pd.DataFrame({
    'variable': meta.column_names,
    'etiqueta': meta.column_labels
})

# Muestra las primeras filas
diccionario.head(20)

In [ ]:
# Buscar una variable por nombre parcial
# Cambia el texto entre comillas para buscar otra variable
diccionario[diccionario['variable'].str.contains('Q1', case=False)]

## 3. Etiquetas de valores (value labels)

Muestra las categorías codificadas de una variable específica.

In [ ]:
# Ver las categorías de una variable
# Cambia 'Q1_2' por cualquier variable que quieras revisar
variable = 'Q1_2'

if variable in meta.variable_value_labels:
    print(f"Etiquetas de valores para '{variable}':")
    for codigo, etiqueta in meta.variable_value_labels[variable].items():
        print(f"  {codigo} -> {etiqueta}")
else:
    print(f"La variable '{variable}' no tiene etiquetas de valor registradas.")

## 4. Recodificaciones estándar

Las siguientes celdas aplican recodificaciones sobre las variables más importantes. Cada recodificación crea una **nueva columna** con sufijo `_rec` para no alterar los datos originales.

### 4.1 Variables de identificación y clasificación

In [ ]:
# --- REGIÓN (COD_REGION) ---
# Reemplaza los códigos numéricos por nombres de región
mapa_region = {
    1: 'Tarapacá', 2: 'Antofagasta', 3: 'Atacama', 4: 'Coquimbo',
    5: 'Valparaíso', 6: "O'Higgins", 7: 'Maule', 8: 'Biobío',
    9: 'Araucanía', 10: 'Los Lagos', 11: 'Aysén', 12: 'Magallanes',
    13: 'Metropolitana', 14: 'Los Ríos', 15: 'Arica y Parinacota', 16: 'Ñuble'
}
df['COD_REGION_rec'] = df['COD_REGION'].map(mapa_region)

# --- ZONA (ZONA) ---
mapa_zona = {1: 'Urbana', 2: 'Rural'}
df['ZONA_rec'] = df['ZONA'].map(mapa_zona)

print("Regiones en la muestra:")
print(df['COD_REGION_rec'].value_counts())

### 4.2 Variables sociodemográficas del entrevistado

In [ ]:
# --- SEXO (Q1_2) ---
mapa_sexo = {1: 'Hombre', 2: 'Mujer'}
df['sexo_rec'] = df['Q1_2'].map(mapa_sexo)

# --- NIVEL EDUCACIONAL (Q1_3) ---
mapa_educ = {
    1: 'Sin educación formal',
    2: 'Básica incompleta',
    3: 'Básica completa',
    4: 'Media CH incompleta',
    5: 'Media TP incompleta',
    6: 'Media CH completa',
    7: 'Media TP completa',
    8: 'Superior técnica incompleta',
    9: 'Superior técnica completa',
    10: 'Superior universitaria incompleta',
    11: 'Superior universitaria completa'
}
df['educ_rec'] = df['Q1_3'].map(mapa_educ)

# --- NIVEL EDUCACIONAL AGRUPADO (3 grupos) ---
mapa_educ_grupo = {
    1: 'Básica o menos', 2: 'Básica o menos', 3: 'Básica o menos',
    4: 'Media', 5: 'Media', 6: 'Media', 7: 'Media',
    8: 'Superior', 9: 'Superior', 10: 'Superior', 11: 'Superior'
}
df['educ_grupo_rec'] = df['Q1_3'].map(mapa_educ_grupo)

# --- TRAMO DE EDAD (Q1_1) ---
# Q1_1 es la edad como número continuo; se crean tramos quinquenales
bins = [0, 17, 29, 44, 59, 200]
labels = ['Menor de 18', '18-29', '30-44', '45-59', '60 y más']
df['tramo_edad_rec'] = pd.cut(df['Q1_1'], bins=bins, labels=labels, right=True)

# --- ACTIVIDAD PRINCIPAL (Q2) ---
mapa_actividad = {
    1: 'Trabajador independiente',
    2: 'Empleador/patrón',
    3: 'Empleado dependiente',
    4: 'Familiar no remunerado',
    5: 'FFAA y de orden',
    6: 'Cesante',
    7: 'Jubilado/pensionado',
    8: 'Estudiante',
    9: 'Labores del hogar'
}
df['actividad_rec'] = df['Q2'].map(mapa_actividad)

# --- NIVEL SOCIOECONÓMICO (A11 / Q1_4) ---
mapa_gse = {1: 'E', 2: 'D', 3: 'C3', 4: 'C2', 5: 'C1', 6: 'AB'}
df['gse_jh_rec'] = df['A11'].map(mapa_gse)   # GSE del jefe de hogar
df['gse_rec'] = df['Q1_4'].map({               # GSE del entrevistado
    1: 'E', 2: 'D', 3: 'C3', 4: 'C2', 5: 'C1', 6: 'AB', 7: 'Sin trabajo'
})

print("Distribución por sexo:")
print(df['sexo_rec'].value_counts())
print("\nDistribución por tramo de edad:")
print(df['tramo_edad_rec'].value_counts().sort_index())

### 4.3 Acceso a Internet en el hogar

In [ ]:
# --- ACCESO A INTERNET EN EL HOGAR (P1) ---
# 1=Sí, 2=No
mapa_si_no = {1: 'Sí', 2: 'No'}
df['acceso_internet_hogar_rec'] = df['P1'].map(mapa_si_no)

# --- TIPO DE ACCESO FIJO (P10) ---
mapa_tipo_acceso = {
    1: 'ADSL', 2: 'Cable/Módem', 3: 'Fibra óptica',
    4: 'Inalámbrica', 5: 'Satelital', 31: 'WiFi',
    32: 'Antena', 33: 'Banda ancha', 34: 'Acceso telefónico', 88: 'No sabe'
}
df['tipo_acceso_fijo_rec'] = df['P10'].map(mapa_tipo_acceso)

# --- VELOCIDAD CONTRATADA (P11_3) ---
mapa_velocidad = {
    1: 'Hasta 10 Mbps',
    2: 'Más de 10 a 100 Mbps',
    3: 'Más de 100 a 500 Mbps',
    4: 'Más de 500 Mbps a 1 Gbps',
    5: 'Más de 1 Gbps',
    99: 'NS/NR'
}
df['velocidad_rec'] = df['P11_3'].map(mapa_velocidad)

# --- TIPO DE PLAN (P12_2) ---
mapa_plan = {
    1: 'Banda ancha desnuda',
    2: 'BA + TV Cable',
    3: 'BA + Telefonía fija',
    4: 'Triple pack (BA+TV+Tel)',
    5: 'Otros planes'
}
df['tipo_plan_rec'] = df['P12_2'].map(mapa_plan)

print("Acceso a Internet en el hogar:")
print(df['acceso_internet_hogar_rec'].value_counts())

### 4.4 Uso de Internet (variables del entrevistado)

In [ ]:
# --- USO DE COMPUTADOR (Q5) ---
df['uso_computador_rec'] = df['Q5'].map({1: 'Sí', 2: 'No'})

# --- USO DE SMARTPHONE (Q7) ---
df['uso_smartphone_rec'] = df['Q7'].map({1: 'Sí', 2: 'No'})

# --- ÚLTIMA VEZ QUE USÓ INTERNET (Q9) ---
mapa_ultimo_uso = {
    1: 'Hoy',
    2: 'Entre 2 y 3 días',
    3: 'Entre 3 y 7 días',
    4: 'Entre 1 y 4 semanas',
    5: 'Más de 4 semanas',
    6: 'Más de 12 meses',
    7: 'Nunca'
}
df['ultimo_uso_internet_rec'] = df['Q9'].map(mapa_ultimo_uso)

# --- FRECUENCIA DE USO (Q10) ---
mapa_frecuencia = {
    1: 'Todos los días',
    2: 'Varias veces por semana',
    3: 'Al menos una vez al mes',
    4: 'Menos de una vez al mes'
}
df['frecuencia_internet_rec'] = df['Q10'].map(mapa_frecuencia)

# --- TIEMPO DIARIO CONECTADO (Q11) ---
mapa_tiempo = {
    1: 'Menos de 1 hora',
    2: 'Entre 1 y 2 horas',
    3: 'Entre 2 y 4 horas',
    4: 'Más de 4 horas'
}
df['tiempo_diario_rec'] = df['Q11'].map(mapa_tiempo)

print("Frecuencia de uso de Internet:")
print(df['frecuencia_internet_rec'].value_counts())

### 4.5 Percepciones y seguridad

In [ ]:
# --- NIVEL DE PROTECCIÓN PERCIBIDA (Q31) ---
mapa_proteccion = {
    1: 'Muy protegido',
    2: 'Protegido',
    3: 'Desprotegido',
    4: 'Muy desprotegido',
    99: 'NS/NR'
}
df['proteccion_percibida_rec'] = df['Q31'].map(mapa_proteccion)

# --- INTERNET MEJORA CALIDAD DE VIDA (Q25) ---
df['internet_mejora_vida_rec'] = df['Q25'].map({1: 'Sí', 2: 'No'})

# --- INTERNET FACILITA ACTIVIDAD LABORAL (Q23) ---
df['internet_facilita_trabajo_rec'] = df['Q23'].map({1: 'Sí', 2: 'No'})

print("Percepción de protección:")
print(df['proteccion_percibida_rec'].value_counts())

### 4.6 Variables de razones de no acceso

Las variables P13_X y Q34_X son de **selección múltiple** (1=mencionada, vacío=no mencionada). No requieren recodificación de etiquetas, pero conviene renombrarlas para facilitar el análisis.

In [ ]:
# Frecuencia de mención de cada razón de no acceso fijo (P13_X)
# Selecciona solo las columnas P13 que son indicadoras (1=sí mencionó)
cols_p13 = [c for c in df.columns if c.startswith('P13_') and c != 'P13_OTRA']

# Cuenta cuántas veces fue marcada cada razón
# (cada columna tiene 1.0 si fue marcada, NaN si no)
conteo_p13 = df[cols_p13].notna().sum().sort_values(ascending=False)
print("Razones de no acceso a internet fijo (P13):")
print(conteo_p13)

## 5. Tratamiento de valores especiales

Algunas variables usan **9999999** como código de NS/NR en variables numéricas (montos, pagos). Conviene reemplazarlos por NaN.

In [ ]:
# Variables numéricas con 9999999 como NS/NR
# P11 (pago mensual), Q7_4, P17_X, P19_X, Q40_X, Q42, Q42_1
cols_numericas_nsnr = [
    'P11', 'Q7_4',
    'P17_1', 'P17_2', 'P17_3', 'P17_4', 'P17_5',
    'P19_1', 'P19_2', 'P19_3', 'P19_4',
    'Q40_1', 'Q40_2', 'Q40_3', 'Q40_4', 'Q40_5',
    'Q42', 'Q42_1'
]

# Reemplaza 9999999 por NaN en las columnas que existen en el DataFrame
for col in cols_numericas_nsnr:
    if col in df.columns:
        df[col] = df[col].replace(9999999, np.nan)

print("Valores 9999999 reemplazados por NaN en variables numéricas.")
print(f"Ejemplo - P11 (pago mensual), promedio: ${df['P11'].mean():,.0f}")

## 6. Resumen del DataFrame recodificado

In [ ]:
# Ver todas las columnas nuevas (con sufijo _rec)
cols_rec = [c for c in df.columns if c.endswith('_rec')]
print(f"Columnas recodificadas creadas: {len(cols_rec)}")
print(cols_rec)

In [ ]:
# Vista rápida del perfil sociodemográfico de la muestra
resumen = {
    'Sexo': df['sexo_rec'].value_counts(normalize=True).mul(100).round(1),
    'Tramo edad': df['tramo_edad_rec'].value_counts(normalize=True).mul(100).round(1),
    'Zona': df['ZONA_rec'].value_counts(normalize=True).mul(100).round(1),
}

for nombre, tabla in resumen.items():
    print(f"\n{nombre} (%):")
    print(tabla)

## 7. Función auxiliar: frecuencia con factor de expansión

Para análisis representativos, usa siempre `FE_PERSONAS` o `FE_HOGAR` según la unidad de análisis.

In [ ]:
def frecuencia_ponderada(df, variable, factor='FE_PERSONAS'):
    """
    Calcula la distribución porcentual de una variable usando factor de expansión.
    
    Parámetros:
    - df: DataFrame
    - variable: nombre de la columna a tabular
    - factor: 'FE_PERSONAS' para análisis de personas, 'FE_HOGAR' para hogares
    """
    # Suma ponderada por categoría
    tabla = df.groupby(variable)[factor].sum()
    
    # Calcula porcentaje
    tabla_pct = (tabla / tabla.sum() * 100).round(1)
    
    # Une frecuencia absoluta y porcentaje
    resultado = pd.DataFrame({
        'N_expandido': tabla.astype(int),
        'Porcentaje': tabla_pct
    })
    
    return resultado

# Ejemplo de uso: distribución por sexo ponderada
print("Distribución por sexo (ponderada):")
frecuencia_ponderada(df, 'sexo_rec')

In [ ]:
# Ejemplo: acceso a internet por zona, ponderado
print("Acceso a Internet en el hogar por zona (ponderado):")
df.groupby(['ZONA_rec', 'acceso_internet_hogar_rec'])['FE_HOGAR'].sum().unstack().apply(lambda x: x / x.sum() * 100, axis=1).round(1)

## 8. Renombrado de variables

Esta sección renombra todas las columnas del DataFrame para hacer el código más legible.

Se aplica en **dos niveles**:
- **Nombres cortos** (~50 variables principales): escritos manualmente, fáciles de recordar.
- **Nombres desde etiqueta** (resto): generados automáticamente a partir del texto de la etiqueta SPSS.

> **Nota:** ejecuta esta sección **después** de aplicar las recodificaciones de la sección 4, ya que esas celdas usan los nombres originales.

In [ ]:
import unicodedata
import re

# ── Función auxiliar ──────────────────────────────────────────
def nombre_desde_etiqueta(texto, max_largo=45):
    """Convierte una etiqueta SPSS en nombre Python válido (snake_case)."""
    if not texto:
        return None
    # Quita acentos
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = texto.lower()
    # Quita texto entre corchetes (pregunta padre)
    texto = re.sub(r'\[.*?\]', '', texto)
    # Quita numeración inicial tipo '1.- '
    texto = re.sub(r'^\d+\.\-?\s*', '', texto.strip())
    # Solo alfanuméricos
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    texto = re.sub(r'\s+', '_', texto.strip())
    return texto[:max_largo].rstrip('_')


# ── Nombres cortos para variables principales ─────────────────
nombres_cortos = {
    # Identificación y clasificación
    'REGISTRO':      'id',
    'FECHAFIN':      'fecha_fin',
    'COD_REGION':    'region',
    'COMUNA_DEF':    'comuna',
    'ZONA':          'zona',

    # Hogar - perfil
    'A9':            'parentesco_jh',
    'A10':           'educ_jh',
    'A11':           'gse_jh',
    'A12_1':         'ingreso_hogar',

    # Entrevistado - perfil
    'Q1':            'parentesco',
    'Q1_1':          'edad',
    'Q1_2':          'sexo',
    'Q1_3':          'educ',
    'Q1_4':          'gse',
    'Q2':            'actividad',

    # Acceso a internet en el hogar
    'P1':            'acceso_internet_hogar',
    'P2':            'n_smartphones_hogar',
    'P2_1':          'n_computadores_hogar',
    'P10':           'tipo_acceso_fijo',
    'P11':           'pago_mensual_internet',
    'P11_3':         'velocidad_contratada',
    'P11_4':         'calidad_acceso',
    'P11_5':         'cuota_mensual_gb',
    'P12_2':         'tipo_plan',
    'P12_1':         'plan_movil_tipo',
    'P14':           'razon_no_acceso_principal',
    'P15':           'disposicion_contratar_fijo',

    # Uso individual de internet
    'Q5':            'uso_computador',
    'Q7':            'uso_smartphone',
    'Q7_1':          'smartphone_propio',
    'Q7_3':          'plan_movil_tipo_ind',
    'Q7_4':          'pago_mensual_movil',
    'Q9':            'ultimo_uso_internet',
    'Q10':           'frecuencia_internet',
    'Q11':           'tiempo_diario_internet',
    'Q13':           'tipo_acceso_mas_usado',
    'Q14':           'uso_internet_hogar',
    'Q15':           'frecuencia_internet_hogar',
    'Q16':           'tiempo_diario_hogar',
    'Q17':           'uso_internet_fuera_hogar',
    'Q18':           'frecuencia_fuera_hogar',
    'Q19':           'tiempo_diario_fuera_hogar',

    # Percepciones
    'Q23':           'internet_facilita_trabajo',
    'Q25':           'internet_mejora_vida',
    'Q27':           'ultima_compra_online',
    'Q31':           'percepcion_proteccion',
    'Q30_1':         'reg_control_legal',
    'Q30_2':         'reg_control_familia',
    'Q30_3':         'reg_autocontrol',

    # Factores de expansión
    'FE_HOGAR':      'fe_hogar',
    'FE_PERSONAS':   'fe_personas',
    'POND_HOGAR':    'pond_hogar',
    'POND_PERSONAS': 'pond_personas',
}

# Solo renombra las que existen en el DataFrame
nombres_cortos_validos = {k: v for k, v in nombres_cortos.items() if k in df.columns}
df = df.rename(columns=nombres_cortos_validos)
print(f"Nombres cortos aplicados: {len(nombres_cortos_validos)}")

In [ ]:
# ── Renombrado automático para el resto de variables ──────────
# Trabaja sobre la lista de nombres originales guardada en meta
vars_ya_renombradas = set(nombres_cortos_validos.keys())
nombres_usados = set(df.columns)  # incluye nombres cortos + columnas _rec ya creadas

renombrado_auto = {}

for col_orig, label in zip(meta.column_names, meta.column_labels):
    # Salta las que ya tienen nombre corto
    if col_orig in vars_ya_renombradas:
        continue
    # Salta las que ya no existen en el DataFrame (ya renombradas)
    # Busca el nombre actual de la columna (puede haber cambiado)
    col_actual = nombres_cortos_validos.get(col_orig, col_orig)
    if col_actual not in df.columns:
        continue

    # Genera nombre desde etiqueta
    nuevo = nombre_desde_etiqueta(label) if label else col_orig.lower()
    if not nuevo:
        nuevo = col_orig.lower()

    # Desambigua si ya existe
    candidato = nuevo
    sufijo = 2
    while candidato in nombres_usados:
        candidato = f"{nuevo}_{sufijo}"
        sufijo += 1

    renombrado_auto[col_actual] = candidato
    nombres_usados.add(candidato)

df = df.rename(columns=renombrado_auto)
print(f"Nombres automáticos aplicados: {len(renombrado_auto)}")
print(f"Total columnas en el DataFrame: {len(df.columns)}")

In [ ]:
# ── Diccionario de referencia completo ───────────────────────
# Permite buscar cualquier variable por su nombre original o nuevo

todos = {**nombres_cortos_validos, **renombrado_auto}

# Construye tabla con nombre original, nombre nuevo y etiqueta SPSS
etiquetas_dict = dict(zip(meta.column_names, meta.column_labels))

diccionario_renombrado = pd.DataFrame({
    'original':  list(todos.keys()),
    'nuevo':     list(todos.values()),
    'etiqueta':  [etiquetas_dict.get(v, '') for v in todos.keys()]
})

print(f"Variables en el diccionario: {len(diccionario_renombrado)}")
diccionario_renombrado.head(20)

In [ ]:
# ── Búsqueda en el diccionario ────────────────────────────────
# Busca por nombre original, nombre nuevo o fragmento de etiqueta
# Cambia el texto entre comillas para buscar otra variable

busqueda = 'whatsapp'

resultado = diccionario_renombrado[
    diccionario_renombrado['original'].str.contains(busqueda, case=False) |
    diccionario_renombrado['nuevo'].str.contains(busqueda, case=False) |
    diccionario_renombrado['etiqueta'].str.contains(busqueda, case=False, na=False)
]
resultado